# K-FAC FROST Decoding

This notebook keeps the token-level shortlist decoder from the paper, uses a K-FAC geometric scorer, and adds evaluation graphs for accuracy, F1, ROC-AUC, response length, latency, and beta tradeoffs.

Structural and provenance terms stay disabled in v1; their hooks remain as zero-valued stubs so the public API matches the paper.


In [ ]:
from datasets import load_dataset
import gc
import hashlib
import json
import math
import re
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.set_grad_enabled(True)

# Data

dataset = load_dataset('gsm8k', 'main')
data = dataset['train'].select(range(1000))

# Decode / scoring defaults

MAX_SEQUENCE_LENGTH = 512
MAX_NEW_TOKENS = 32
SHORTLIST_K = 5
DEFAULT_BETA = 0.5
EVAL_LIMIT = 8
SMOKE_TEST_LIMIT = 1

# K-FAC defaults

KFAC_RANK = 64
KFAC_DAMPING = 1e-3
KFAC_LAYER_LIMIT = 16
PROXY_CALIBRATION_SAMPLES = 8

# Composite energy weights (v1: geometry only)

LAMBDA_GEOM = 1.0
LAMBDA_STRUCT = 0.0
LAMBDA_PROV = 0.0

EPS = 1e-6
ANSWER_RE = re.compile(r'-?\d+(?:,\d{3})*(?:\.\d+)?')


def model_device(model):
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device('cpu')


def move_batch_to_device(batch, device):
    return {k: v.to(device) for k, v in batch.items()}


def stable_seed(text):
    return int(hashlib.sha256(text.encode('utf-8')).hexdigest()[:8], 16)


def extract_number(text):
    if text is None:
        return None
    text = str(text)
    if '####' in text:
        text = text.split('####')[-1]
    matches = ANSWER_RE.findall(text.replace('$', ''))
    if not matches:
        return None
    return matches[-1].replace(',', '')


def gsm8k_correct(prediction, gold):
    pred = extract_number(prediction)
    truth = extract_number(gold)
    if pred is None or truth is None:
        return False
    try:
        return abs(float(pred) - float(truth)) < 1e-6
    except ValueError:
        return pred == truth


def response_length(text, tokenizer):
    return len(tokenizer.encode(text, add_special_tokens=False))


In [ ]:
teacher_name = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B'
student_name = 'mistralai/Mistral-7B-Instruct-v0.3'

teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_name)
student_tokenizer = AutoTokenizer.from_pretrained(student_name)

if teacher_tokenizer.pad_token is None:
    teacher_tokenizer.pad_token = teacher_tokenizer.eos_token
if student_tokenizer.pad_token is None:
    student_tokenizer.pad_token = student_tokenizer.eos_token

teacher = AutoModelForCausalLM.from_pretrained(
    teacher_name,
    torch_dtype=torch.float16,
    device_map='auto',
)

student = AutoModelForCausalLM.from_pretrained(
    student_name,
    torch_dtype=torch.float16,
    device_map='auto',
)

teacher.eval()
student.eval()

teacher_input_device = model_device(teacher)
student_input_device = model_device(student)

print(f'Teacher device: {teacher_input_device}')
print(f'Student device: {student_input_device}')


In [ ]:
def encode_prompt_response(tokenizer, prompt, response, device, max_length=MAX_SEQUENCE_LENGTH):
    full_text = prompt + response
    enc = tokenizer(
        full_text,
        return_tensors='pt',
        truncation=True,
        max_length=max_length,
    )
    enc = move_batch_to_device(enc, device)
    prompt_len = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=max_length,
    )['input_ids'].shape[-1]
    labels = enc['input_ids'].clone()
    if prompt_len < labels.shape[-1]:
        labels[:, :prompt_len] = -100
    labels[enc['attention_mask'] == 0] = -100
    return enc, labels, prompt_len


@dataclass
class KFACLayerState:
    name: str
    module: nn.Linear
    input_proj: torch.Tensor
    output_proj: torch.Tensor
    a_cov: torch.Tensor
    g_cov: torch.Tensor
    a_inv: torch.Tensor
    g_inv: torch.Tensor
    a_count: int = 0
    g_count: int = 0
    cached_a_proj: Optional[torch.Tensor] = None


@dataclass
class ProxyKFACState:
    name: str
    model: nn.Module
    tokenizer: object
    layers: Dict[str, KFACLayerState] = field(default_factory=dict)
    handles: List = field(default_factory=list)
    calibrating: bool = False


def select_linear_modules(model, limit=KFAC_LAYER_LIMIT):
    linear_items = []
    for name, module in model.named_modules():
        if not name:
            continue
        if not isinstance(module, nn.Linear):
            continue
        if 'lm_head' in name or 'embed_tokens' in name:
            continue
        linear_items.append((name, module))
    if limit is not None:
        linear_items = linear_items[-limit:]
    return linear_items


def build_projection(in_dim, rank, device, seed):
    generator = torch.Generator(device=device)
    generator.manual_seed(seed)
    proj = torch.randn(in_dim, rank, generator=generator, device=device, dtype=torch.float32)
    return proj / math.sqrt(rank)


def build_proxy_state(name, model, tokenizer):
    layers = {}
    for layer_name, module in select_linear_modules(model):
        device = module.weight.device
        in_rank = min(KFAC_RANK, module.in_features)
        out_rank = min(KFAC_RANK, module.out_features)
        seed = stable_seed(f'{name}:{layer_name}')
        input_proj = build_projection(module.in_features, in_rank, device, seed)
        output_proj = build_projection(module.out_features, out_rank, device, seed + 1)
        a_cov = torch.zeros(in_rank, in_rank, device=device, dtype=torch.float32)
        g_cov = torch.zeros(out_rank, out_rank, device=device, dtype=torch.float32)
        eye_a = torch.eye(in_rank, device=device, dtype=torch.float32)
        eye_g = torch.eye(out_rank, device=device, dtype=torch.float32)
        layers[layer_name] = KFACLayerState(
            name=layer_name,
            module=module,
            input_proj=input_proj,
            output_proj=output_proj,
            a_cov=a_cov,
            g_cov=g_cov,
            a_inv=eye_a.clone(),
            g_inv=eye_g.clone(),
        )
    return ProxyKFACState(name=name, model=model, tokenizer=tokenizer, layers=layers)


def register_kfac_hooks(proxy):
    for layer_name, state in proxy.layers.items():
        module = state.module

        def forward_hook(mod, inputs, output, state=state, proxy=proxy):
            if not proxy.calibrating or not inputs:
                return
            activations = inputs[0].detach()
            activations = activations.reshape(-1, activations.shape[-1])
            activations = activations.to(device=state.input_proj.device, dtype=torch.float32)
            state.cached_a_proj = activations @ state.input_proj

        def backward_hook(mod, grad_input, grad_output, state=state, proxy=proxy):
            if not proxy.calibrating or state.cached_a_proj is None:
                return
            gradients = grad_output[0].detach()
            gradients = gradients.reshape(-1, gradients.shape[-1])
            gradients = gradients.to(device=state.output_proj.device, dtype=torch.float32)
            g_proj = gradients @ state.output_proj
            state.a_cov += state.cached_a_proj.T @ state.cached_a_proj
            state.g_cov += g_proj.T @ g_proj
            state.a_count += int(state.cached_a_proj.shape[0])
            state.g_count += int(g_proj.shape[0])
            state.cached_a_proj = None

        proxy.handles.append(module.register_forward_hook(forward_hook))
        proxy.handles.append(module.register_full_backward_hook(backward_hook))


def clear_proxy_hooks(proxy):
    for handle in proxy.handles:
        handle.remove()
    proxy.handles.clear()


def safe_inverse(matrix):
    eye = torch.eye(matrix.shape[0], device=matrix.device, dtype=matrix.dtype)
    damped = matrix + KFAC_DAMPING * eye
    try:
        return torch.linalg.inv(damped)
    except RuntimeError:
        return torch.linalg.pinv(damped)


def finalize_proxy(proxy):
    for state in proxy.layers.values():
        a_count = max(1, state.a_count)
        g_count = max(1, state.g_count)
        a_cov = state.a_cov / a_count
        g_cov = state.g_cov / g_count
        state.a_inv = safe_inverse(a_cov)
        state.g_inv = safe_inverse(g_cov)


def calibrate_proxy(proxy, calibration_pairs, max_samples=PROXY_CALIBRATION_SAMPLES):
    device = model_device(proxy.model)
    proxy.model.eval()
    proxy.calibrating = True
    proxy.model.zero_grad(set_to_none=True)
    for prompt, response in calibration_pairs[:max_samples]:
        enc, labels, _ = encode_prompt_response(proxy.tokenizer, prompt, response, device)
        if (labels != -100).sum().item() == 0:
            continue
        outputs = proxy.model(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask'],
            labels=labels,
        )
        loss = outputs.loss.float()
        proxy.model.zero_grad(set_to_none=True)
        loss.backward()
    proxy.calibrating = False
    finalize_proxy(proxy)


In [ ]:
calibration_pairs = [
    (
        f"Question: {example['question']}\nAnswer:",
        example['answer'],
    )
    for example in data.select(range(min(PROXY_CALIBRATION_SAMPLES, len(data))))
]

proxy_specs = [
    ('student', student, student_tokenizer),
]

proxy_states = []
for proxy_name, proxy_model, proxy_tokenizer in proxy_specs:
    proxy = build_proxy_state(proxy_name, proxy_model, proxy_tokenizer)
    register_kfac_hooks(proxy)
    calibrate_proxy(proxy, calibration_pairs)
    clear_proxy_hooks(proxy)
    proxy_states.append(proxy)
    print(f"Calibrated {proxy.name}: {len(proxy.layers)} K-FAC layers")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
def structural_energy(*args, **kwargs):
    return 0.0


def provenance_energy(*args, **kwargs):
    return 0.0


def score_proxy_candidate(proxy, prompt, full_candidate_text, prompt_len=None):
    device = model_device(proxy.model)
    enc = proxy.tokenizer(
        full_candidate_text,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_SEQUENCE_LENGTH,
    )
    enc = move_batch_to_device(enc, device)
    if prompt_len is None:
        prompt_len = proxy.tokenizer(
            prompt,
            return_tensors='pt',
            truncation=True,
            max_length=MAX_SEQUENCE_LENGTH,
        )['input_ids'].shape[-1]
    labels = enc['input_ids'].clone()
    if prompt_len < labels.shape[-1]:
        labels[:, :prompt_len] = -100
    labels[enc['attention_mask'] == 0] = -100
    if (labels != -100).sum().item() == 0:
        return 0.0

    proxy.model.zero_grad(set_to_none=True)
    outputs = proxy.model(
        input_ids=enc['input_ids'],
        attention_mask=enc['attention_mask'],
        labels=labels,
    )
    loss = outputs.loss.float()
    loss.backward()

    energy = 0.0
    with torch.no_grad():
        for state in proxy.layers.values():
            grad = state.module.weight.grad
            if grad is None:
                continue
            grad = grad.detach().float()
            dW_proj = state.output_proj.T @ grad @ state.input_proj
            layer_energy = torch.trace(state.g_inv @ dW_proj @ state.a_inv @ dW_proj.T)
            energy += float(layer_energy.item())
            if state.module.bias is not None and state.module.bias.grad is not None:
                bias_grad = state.module.bias.grad.detach().float()
                bias_proj = bias_grad @ state.output_proj
                energy += float(torch.dot(bias_proj, state.g_inv @ bias_proj).item())

    proxy.model.zero_grad(set_to_none=True)
    return energy


def score_candidate_over_proxies(prompt, full_candidate_text, prompt_len_cache=None):
    proxy_energies = []
    for proxy in proxy_states:
        cached_prompt_len = None
        if prompt_len_cache is not None:
            cached_prompt_len = prompt_len_cache.get(proxy.name)
        proxy_energies.append(score_proxy_candidate(proxy, prompt, full_candidate_text, prompt_len=cached_prompt_len))
    if not proxy_energies:
        return 0.0, []
    return float(np.mean(proxy_energies)), proxy_energies


def teacher_generate(prompt, max_new_tokens=MAX_NEW_TOKENS, sample=False, return_stats=False):
    inputs = teacher_tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_SEQUENCE_LENGTH,
    )
    inputs = move_batch_to_device(inputs, teacher_input_device)
    start = time.perf_counter()
    with torch.no_grad():
        outputs = teacher.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=sample,
            pad_token_id=teacher_tokenizer.eos_token_id,
        )
    elapsed = time.perf_counter() - start
    text = teacher_tokenizer.decode(outputs[0], skip_special_tokens=True)
    stats = {
        'elapsed_sec': float(elapsed),
        'num_tokens': int(outputs.shape[-1]),
        'generated_tokens': int(outputs.shape[-1] - inputs['input_ids'].shape[-1]),
    }
    if return_stats:
        return text, stats
    return text


def frost_generate(prompt, max_new_tokens=MAX_NEW_TOKENS, top_k=SHORTLIST_K, beta=DEFAULT_BETA, sample=False, return_stats=False):
    if not proxy_states:
        raise RuntimeError('proxy_states is empty; calibrate at least one proxy before decoding.')

    input_ids = teacher_tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_SEQUENCE_LENGTH,
    )['input_ids'].to(teacher_input_device)
    decoded_text = prompt
    step_records = []
    start = time.perf_counter()

    for step in range(max_new_tokens):
        step_start = time.perf_counter()
        with torch.no_grad():
            logits = teacher(input_ids).logits[:, -1, :]

        topk = torch.topk(logits, k=top_k)
        candidate_ids = topk.indices[0]
        candidate_texts = []
        candidate_geom = []
        candidate_proxy_breakdown = []
        prompt_len_cache = {
            proxy.name: proxy.tokenizer(
                prompt,
                return_tensors='pt',
                truncation=True,
                max_length=MAX_SEQUENCE_LENGTH,
            )['input_ids'].shape[-1]
            for proxy in proxy_states
        }

        for token_id in candidate_ids:
            next_ids = torch.cat([input_ids, token_id.view(1, 1)], dim=1)
            full_candidate_text = teacher_tokenizer.decode(next_ids[0], skip_special_tokens=True)
            geom_score, per_proxy = score_candidate_over_proxies(prompt, full_candidate_text, prompt_len_cache)
            candidate_texts.append(full_candidate_text)
            candidate_geom.append(float(geom_score))
            candidate_proxy_breakdown.append([float(x) for x in per_proxy])

        geom_tensor = torch.tensor(candidate_geom, dtype=logits.dtype, device=logits.device)
        if geom_tensor.numel() > 1 and torch.isfinite(geom_tensor).all() and geom_tensor.std() > EPS:
            geom_tensor = (geom_tensor - geom_tensor.mean()) / (geom_tensor.std() + EPS)
        else:
            geom_tensor = torch.zeros_like(geom_tensor)

        structural_tensor = torch.tensor(
            [structural_energy(prompt, text) for text in candidate_texts],
            dtype=logits.dtype,
            device=logits.device,
        )
        provenance_tensor = torch.tensor(
            [provenance_energy(prompt, text) for text in candidate_texts],
            dtype=logits.dtype,
            device=logits.device,
        )

        composite_energy = (
            LAMBDA_GEOM * geom_tensor
            + LAMBDA_STRUCT * structural_tensor
            + LAMBDA_PROV * provenance_tensor
        )

        shortlist_logits = logits[0, candidate_ids] - beta * composite_energy
        if sample:
            shortlist_probs = torch.softmax(shortlist_logits, dim=-1)
            shortlist_choice = torch.multinomial(shortlist_probs, 1)
        else:
            shortlist_choice = torch.argmax(shortlist_logits, dim=-1, keepdim=True)
        next_token = candidate_ids[shortlist_choice]

        input_ids = torch.cat([input_ids, next_token.view(1, 1)], dim=1)
        decoded_text = teacher_tokenizer.decode(input_ids[0], skip_special_tokens=True)

        chosen_index = int(shortlist_choice.item())
        chosen_token_id = int(next_token.item())
        chosen_energy = float(candidate_geom[chosen_index])
        step_records.append({
            'step': int(step),
            'candidates': [teacher_tokenizer.decode([int(t.item())], skip_special_tokens=True) for t in candidate_ids],
            'candidate_texts': candidate_texts,
            'candidate_geom': candidate_geom,
            'candidate_proxy_breakdown': candidate_proxy_breakdown,
            'candidate_structural': [float(x) for x in structural_tensor.tolist()],
            'candidate_provenance': [float(x) for x in provenance_tensor.tolist()],
            'chosen_token_id': chosen_token_id,
            'chosen_token_text': teacher_tokenizer.decode([chosen_token_id], skip_special_tokens=True),
            'chosen_energy': chosen_energy,
            'step_time_sec': float(time.perf_counter() - step_start),
        })

        if teacher_tokenizer.eos_token_id is not None and chosen_token_id == teacher_tokenizer.eos_token_id:
            break

    elapsed = time.perf_counter() - start
    chosen_energies = [record['chosen_energy'] for record in step_records if record['chosen_energy'] is not None]
    diagnostics = {
        'elapsed_sec': float(elapsed),
        'num_steps': int(len(step_records)),
        'mean_step_time_sec': float(np.mean([record['step_time_sec'] for record in step_records])) if step_records else 0.0,
        'mean_candidate_energy': float(np.mean([np.mean(record['candidate_geom']) for record in step_records])) if step_records else 0.0,
        'mean_chosen_energy': float(np.mean(chosen_energies)) if chosen_energies else 0.0,
        'step_records': step_records,
    }
    if return_stats:
        return decoded_text, diagnostics
    return decoded_text


In [ ]:
evaluation_examples = data.select(range(min(EVAL_LIMIT, len(data))))
results = []

for example in tqdm(evaluation_examples):
    prompt = f"Question: {example['question']}\nAnswer:"

    teacher_out, teacher_stats = teacher_generate(prompt, return_stats=True)
    frost_out, frost_stats = frost_generate(prompt, beta=DEFAULT_BETA, top_k=SHORTLIST_K, return_stats=True)

    teacher_correct = gsm8k_correct(teacher_out, example['answer'])
    frost_correct = gsm8k_correct(frost_out, example['answer'])

    row = {
        'question': example['question'],
        'ground_truth': example['answer'],
        'beta': float(DEFAULT_BETA),
        'top_k': int(SHORTLIST_K),
        'teacher': teacher_out,
        'frost': frost_out,
        'teacher_correct': bool(teacher_correct),
        'frost_correct': bool(frost_correct),
        'teacher_length': int(response_length(teacher_out, teacher_tokenizer)),
        'frost_length': int(response_length(frost_out, teacher_tokenizer)),
        'teacher_time_sec': float(teacher_stats['elapsed_sec']),
        'frost_time_sec': float(frost_stats['elapsed_sec']),
        'frost_mean_candidate_energy': float(frost_stats['mean_candidate_energy']),
        'frost_mean_chosen_energy': float(frost_stats['mean_chosen_energy']),
        'frost_num_steps': int(frost_stats['num_steps']),
        'frost_step_records': frost_stats['step_records'],
    }
    results.append(row)

    print('Teacher:', teacher_out)
    print('FROST:', frost_out)
    print('Teacher correct:', teacher_correct, '| FROST correct:', frost_correct)
    print('-' * 80)

with open('gsm8k_frost_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f'Saved {len(results)} rows to gsm8k_frost_results.json')


In [ ]:
def roc_curve_manual(y_true, y_score):
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)
    if y_true.size == 0 or len(np.unique(y_true)) < 2:
        return None

    order = np.argsort(-y_score)
    y_true = y_true[order]
    y_score = y_score[order]

    positives = int(y_true.sum())
    negatives = int(len(y_true) - positives)
    if positives == 0 or negatives == 0:
        return None

    tps = np.cumsum(y_true)
    fps = np.cumsum(1 - y_true)
    tpr = np.concatenate([[0.0], tps / positives, [1.0]])
    fpr = np.concatenate([[0.0], fps / negatives, [1.0]])
    auc = float(np.trapz(tpr, fpr))
    return fpr, tpr, auc


def threshold_f1_curve(y_true, y_score):
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)
    if y_true.size == 0:
        return np.array([]), np.array([]), np.array([]), np.array([]), 0

    thresholds = np.unique(y_score)
    if thresholds.size == 1:
        thresholds = np.array([thresholds[0] - 1e-9, thresholds[0], thresholds[0] + 1e-9])
    else:
        thresholds = np.concatenate(([thresholds.min() - 1e-9], thresholds, [thresholds.max() + 1e-9]))

    precision_list = []
    recall_list = []
    f1_list = []
    for thr in thresholds:
        y_pred = (y_score >= thr).astype(int)
        tp = int(np.sum((y_pred == 1) & (y_true == 1)))
        fp = int(np.sum((y_pred == 1) & (y_true == 0)))
        fn = int(np.sum((y_pred == 0) & (y_true == 1)))
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        precision_list.append(precision)
        recall_list.append(recall)
        f1_list.append(f1)

    best_idx = int(np.argmax(f1_list)) if f1_list else 0
    return thresholds, np.array(precision_list), np.array(recall_list), np.array(f1_list), best_idx


def summarize_results(rows):
    if not rows:
        return {}

    teacher_correct = np.array([int(r['teacher_correct']) for r in rows], dtype=int)
    frost_correct = np.array([int(r['frost_correct']) for r in rows], dtype=int)
    teacher_length = np.array([r['teacher_length'] for r in rows], dtype=float)
    frost_length = np.array([r['frost_length'] for r in rows], dtype=float)
    teacher_time = np.array([r['teacher_time_sec'] for r in rows], dtype=float)
    frost_time = np.array([r['frost_time_sec'] for r in rows], dtype=float)
    energy_score = -np.array([r['frost_mean_candidate_energy'] for r in rows], dtype=float)

    summary = {
        'teacher_accuracy': float(teacher_correct.mean()),
        'frost_accuracy': float(frost_correct.mean()),
        'teacher_mean_length': float(teacher_length.mean()),
        'frost_mean_length': float(frost_length.mean()),
        'teacher_mean_latency_sec': float(teacher_time.mean()),
        'frost_mean_latency_sec': float(frost_time.mean()),
        'mean_energy_score': float(energy_score.mean()),
    }

    roc = roc_curve_manual(frost_correct, energy_score)
    if roc is not None:
        fpr, tpr, auc = roc
        summary['frost_auc_roc'] = float(auc)
    else:
        summary['frost_auc_roc'] = None

    thresholds, precision, recall, f1, best_idx = threshold_f1_curve(frost_correct, energy_score)
    summary['best_f1'] = float(f1[best_idx]) if f1.size else None
    summary['best_f1_threshold'] = float(thresholds[best_idx]) if thresholds.size else None
    return summary


def plot_results(rows):
    if not rows:
        print('No results to plot.')
        return

    summary = summarize_results(rows)
    teacher_correct = np.array([int(r['teacher_correct']) for r in rows], dtype=int)
    frost_correct = np.array([int(r['frost_correct']) for r in rows], dtype=int)
    teacher_length = np.array([r['teacher_length'] for r in rows], dtype=float)
    frost_length = np.array([r['frost_length'] for r in rows], dtype=float)
    teacher_time = np.array([r['teacher_time_sec'] for r in rows], dtype=float)
    frost_time = np.array([r['frost_time_sec'] for r in rows], dtype=float)
    energy_score = -np.array([r['frost_mean_candidate_energy'] for r in rows], dtype=float)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    labels = ['Teacher', 'FROST']
    accs = [float(teacher_correct.mean()), float(frost_correct.mean())]
    lengths = [float(teacher_length.mean()), float(frost_length.mean())]
    latencies = [float(teacher_time.mean()), float(frost_time.mean())]

    ax = axes[0]
    x = np.arange(len(labels))
    ax.bar(x - 0.18, accs, width=0.36, label='Accuracy', color='#2c7fb8')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel('Accuracy')
    ax.set_ylim(0.0, 1.0)
    ax2 = ax.twinx()
    ax2.plot(x, lengths, marker='o', linewidth=2, color='#d95f0e', label='Mean length')
    ax2.set_ylabel('Mean response length')
    ax.set_title('Accuracy / Length tradeoff')

    ax = axes[1]
    roc = roc_curve_manual(frost_correct, energy_score)
    if roc is not None:
        fpr, tpr, auc = roc
        ax.plot(fpr, tpr, color='#31a354', linewidth=2, label=f'ROC AUC = {auc:.3f}')
        ax.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1)
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title('AUC-ROC on FROST correctness')
        ax.legend(loc='lower right')
    else:
        ax.text(0.5, 0.5, 'ROC curve unavailable\n(single class only)', ha='center', va='center')
        ax.set_axis_off()

    ax = axes[2]
    thresholds, precision, recall, f1, best_idx = threshold_f1_curve(frost_correct, energy_score)
    if thresholds.size:
        ax.plot(thresholds, precision, label='Precision', linewidth=2)
        ax.plot(thresholds, recall, label='Recall', linewidth=2)
        ax.plot(thresholds, f1, label='F1', linewidth=3)
        ax.axvline(thresholds[best_idx], linestyle='--', color='black', alpha=0.6)
        ax.set_xlabel('Energy-score threshold')
        ax.set_ylabel('Score')
        ax.set_title('Threshold sweep')
        ax.legend()
    else:
        ax.text(0.5, 0.5, 'Threshold curve unavailable', ha='center', va='center')
        ax.set_axis_off()

    plt.tight_layout()
    plt.savefig('frost_kfac_metrics.png', dpi=200, bbox_inches='tight')
    plt.show()

    beta_groups = {}
    for row in rows:
        beta_groups.setdefault(row['beta'], []).append(row)

    if len(beta_groups) > 1:
        betas = sorted(beta_groups)
        beta_accuracy = [float(np.mean([int(r['frost_correct']) for r in beta_groups[b]])) for b in betas]
        beta_lengths = [float(np.mean([r['frost_length'] for r in beta_groups[b]])) for b in betas]

        fig, ax1 = plt.subplots(figsize=(8, 5))
        ax1.plot(betas, beta_accuracy, marker='o', linewidth=2, color='#2c7fb8', label='Accuracy')
        ax1.set_xlabel('Beta')
        ax1.set_ylabel('FROST accuracy')
        ax1.set_ylim(0.0, 1.0)
        ax2 = ax1.twinx()
        ax2.plot(betas, beta_lengths, marker='s', linewidth=2, color='#d95f0e', label='Mean length')
        ax2.set_ylabel('Mean response length')
        ax1.set_title('Beta tradeoff curve')
        fig.tight_layout()
        plt.savefig('frost_kfac_beta_tradeoff.png', dpi=200, bbox_inches='tight')
        plt.show()
    else:
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.scatter(teacher_length.mean(), float(teacher_correct.mean()), s=120, label='Teacher', color='#2c7fb8')
        ax.scatter(frost_length.mean(), float(frost_correct.mean()), s=120, label='FROST', color='#d95f0e')
        ax.set_xlabel('Mean response length')
        ax.set_ylabel('Accuracy')
        ax.set_title('Accuracy vs response length')
        ax.legend()
        plt.tight_layout()
        plt.savefig('frost_kfac_accuracy_length.png', dpi=200, bbox_inches='tight')
        plt.show()

    print(json.dumps(summary, indent=2))
    return summary


rows = results if 'results' in globals() else json.loads(Path('gsm8k_frost_results.json').read_text(encoding='utf-8'))
summary = plot_results(rows)
